# Lab 3 — DBSCAN Clusters

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Apply **DBSCAN** density-based clustering on scaled NYSE symbol features.
2. Identify **noise** points labeled **-1**.
3. Compare DBSCAN to K-Means (Lab 1) — no fixed k required.
4. Explore sensitivity to **eps** and **min_samples**.

> **Checkpoints:** **3** clusters · **11** noise points · counts `{-1:11, 0:6, 1:4, 2:4}`

**Companion script:** `../scripts/lab03_dbscan_clusters.py`

## DBSCAN vs K-Means

| | K-Means (Lab 1) | DBSCAN (this lab) |
|---|----------------|-------------------|
| **k** | Must specify k=4 | Discovers clusters from density |
| **Noise** | Every point assigned | Sparse points → label **-1** |
| **Shape** | Spherical clusters | Arbitrary shapes (in principle) |
| **Params** | `n_clusters` | `eps` (radius), `min_samples` |

DBSCAN links to Day 4 proximity ideas — points in **sparse** regions are outliers.

## Syllabus note — Spectral & OPTICS

<!-- cisco-enrich-2026-06 -->

The course also mentions **Spectral Clustering** (graph Laplacian + eigenvectors) and **OPTICS** (ordering points to reveal varying density). We focus on **DBSCAN** in this lab because it is fast to tune in class and explicitly labels **noise** — critical for financial symbol watch lists.

| Algorithm | Best when |
|-----------|----------|
| K-Means | Known k, spherical segments |
| DBSCAN | Unknown k, noise matters |
| Spectral | Non-convex clusters (theory) |
| OPTICS | Varying density (advanced) |

---

## 1. Load and scale symbol features

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])
print(f"symbols: {len(features)}")

---

## 2. Fit DBSCAN (eps=1.2, min_samples=3)

In [ ]:
EPS = 1.2
MIN_SAMPLES = 3

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

unique, counts = np.unique(labels, return_counts=True)
label_counts = dict(zip(unique.tolist(), counts.tolist()))

print("Lab 3 — DBSCAN clusters")
print(f"eps: {EPS}, min_samples: {MIN_SAMPLES}")
print(f"clusters found: {n_clusters}")
print(f"noise points: {n_noise}")
print(f"label counts: {label_counts}")

---

## 3. Symbol assignments

In [ ]:
features = features.copy()
features["dbscan_label"] = labels

display(
    features[["symbol", "avg_close", "volatility", "dbscan_label"]]
    .sort_values("dbscan_label")
    .round(2)
)

Label **-1** = noise — symbols in sparse regions of scaled feature space (not assigned to any cluster).

---

## 4. Visualize (avg_close vs volatility)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
palette = features["dbscan_label"].map(lambda x: "lightgray" if x == -1 else None)
scatter = ax.scatter(
    features["avg_close"],
    features["volatility"],
    c=features["dbscan_label"],
    cmap="tab10",
    s=100,
    alpha=0.85,
)
for _, row in features.iterrows():
    color = "gray" if row["dbscan_label"] == -1 else "black"
    ax.annotate(row["symbol"], (row["avg_close"], row["volatility"]), fontsize=8, color=color, alpha=0.8)
ax.set_xlabel("avg_close")
ax.set_ylabel("volatility")
ax.set_title(f"DBSCAN (eps={EPS}, noise={n_noise})")
fig.colorbar(scatter, ax=ax, label="cluster (-1 = noise)")
plt.tight_layout()
plt.show()

---

## 5. Compare to K-Means (Lab 1)

In [ ]:
kmeans_labels = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)
kmeans_counts = dict(zip(*np.unique(kmeans_labels, return_counts=True)))

compare = pd.DataFrame({
    "method": ["K-Means (k=4)", "DBSCAN"],
    "clusters": [4, n_clusters],
    "noise_or_unassigned": [0, n_noise],
    "label_counts": [str(kmeans_counts), str(label_counts)],
})
display(compare)

K-Means forces every symbol into a cluster; DBSCAN rejects **11** borderline symbols as noise.

---

## 6. Extension — try different eps values

In [ ]:
rows = []
for eps in [0.8, 1.2, 1.5]:
    lbl = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X_scaled)
    n_clust = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_no = int(np.sum(lbl == -1))
    rows.append({"eps": eps, "clusters": n_clust, "noise": n_no})

display(pd.DataFrame(rows))

Smaller **eps** → stricter neighborhoods → more noise. Larger **eps** → fewer, broader clusters.

---

## 7. Checkpoint summary

In [ ]:
assert len(features) == 25
assert n_clusters == 3
assert n_noise == 11
assert label_counts[-1] == 11
assert label_counts[0] == 6 and label_counts[1] == 4 and label_counts[2] == 4
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. When would a portfolio manager prefer DBSCAN noise labels over K-Means assignments?
2. What happens if `min_samples` equals the total number of symbols?
3. How does DBSCAN relate to Day 6 LOF anomaly detection?

**Previous:** [Lab 2 — Elbow method](lab02_elbow_method.ipynb)  
**Next:** [Lab 4 — Cluster metrics](lab04_cluster_metrics.ipynb)